# Project0 Colab Tutorial

This notebook prepares a Google Colab GPU runtime, installs Project0 and its dependencies, starts a local large language model through Ollama, and makes the Project0 Dashboard available through a temporary public link.

## What you will learn

By completing the notebook, you will see how the main parts of the Colab environment work together:

- **Google Colab** supplies the temporary Linux computer and NVIDIA GPU.
- **Ollama** manages and runs the local language model.
- **`qwen2.5:7b`** is the language model used for Project0 reasoning in this notebook.
- **Project0** provides the Dashboard and agent workflows.
- **Cloudflare Quick Tunnel** gives your browser a temporary HTTPS link to the Dashboard running inside Colab.

The request path is:

```text
Your browser → Cloudflare Quick Tunnel → Project0 Dashboard → Ollama → qwen2.5:7b
```

## Before you begin

In Colab, select **Runtime → Change runtime type**, choose an available **GPU** accelerator, and save the setting. Then run the notebook from top to bottom. You may use **Runtime → Run all**; the optional cleanup in Step 8 is disabled by default so the Dashboard remains available.

Colab runtimes are temporary. Installed software, downloaded models, logs, and other files under `/content` disappear when the runtime is deleted.

## Step 1 - Clone or Update Project0

This step retrieves the Project0 source code from its public GitHub repository and makes it the notebook's working directory.

- If `/content/project0` does not exist, the repository is cloned.
- If it already exists, `git pull --ff-only` downloads newer commits without creating a merge commit.
- No GitHub account, access token, or Colab secret is required for the public repository.

When the step succeeds, the final message is `Project0 repository ready.` and subsequent cells operate from `/content/project0`.

If the update cannot proceed, the existing checkout may contain local changes or may no longer have a direct update path. Review the preceding Git message before rerunning the cell.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/project0")
REPO_URL = "https://github.com/pgailinas/project0.git"

if not REPO_DIR.exists():
    print("Cloning Project0...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Project0 already exists. Updating repository...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/project0

print("Project0 repository ready.")



## Step 2 - Prepare the Environment and Install Project0

This step first confirms that the Colab runtime uses Python 3.12 or newer, which Project0 requires.

Google Colab includes the optional `jieba` language-segmentation package. Project0 does not require this package, but MkDocs Material automatically detects and imports it when present. Under Python 3.13, that import produces compatibility warnings during Documentation Agent validation. The cell removes `jieba` to keep the validation results focused on Project0 documentation.

The cell then installs Project0 and its Python dependencies into the current runtime. The editable installation points Python to the source files in `/content/project0`; it does not replace the cloned source with a separate packaged copy. This is useful during testing because the installed application follows the checked-out repository.

Dependency installation can take a few minutes during a new Colab session. The step is complete when `Project0 installation complete.` appears. A Python-version error means the selected Colab runtime is not compatible with the current Project0 requirements.

The cell is safe to rerun. If `jieba` has already been removed, the uninstall command reports that it is not installed and continues.

In [ ]:
import sys
import subprocess

print("Python:", sys.version)

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Project0 requires Python >= 3.12. "
        f"This Colab runtime is Python "
        f"{sys.version_info.major}.{sys.version_info.minor}."
    )

# Colab includes jieba even though Project0 does not require it.
# MkDocs Material detects the optional package and imports it,
# producing Python 3.13 compatibility warnings during validation.
print("\nRemoving unused Colab jieba package...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "jieba",
    ],
    check=False,
)

print("\nInstalling Project0 from the cloned repository...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        "/content/project0",
    ],
    check=True,
)

print("\nProject0 installation complete.")



## Step 3 - Verify the Colab GPU

Large language models perform their calculations much faster on a supported graphics processor. This step asks PyTorch whether CUDA—the NVIDIA GPU computing interface—is available, then reports the assigned GPU and its total video memory (VRAM).

Expected results include:

- `CUDA available: True`
- A GPU model name
- A positive VRAM value

If CUDA is unavailable, select **Runtime → Change runtime type → GPU** and restart the notebook. The exact GPU model can vary between Colab sessions and account types.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

gpu_index = 0
gpu_name = torch.cuda.get_device_name(gpu_index)
gpu_props = torch.cuda.get_device_properties(gpu_index)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print("GPU:", gpu_name)
print(f"VRAM: {total_vram_gb:.1f} GB")

## Step 4 - Install and Start Ollama

Ollama is the local model runtime used by this notebook. It downloads, loads, and executes language models and exposes a local API that Project0 can call. It is not the language model itself.

This step installs the `zstd` compression dependency and Ollama, starts `ollama serve` as a background process, and waits for its API at `127.0.0.1:11434`. That address is private to the Colab runtime and is not exposed directly to the Internet. Ollama output is recorded in `/content/ollama.log`.

Successful output includes `Ollama server: RUNNING` and an HTTP status of `200`. If Ollama does not become ready, the cell reports its startup log to help identify the failure.

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import urllib.request

OLLAMA_LOG = "/content/ollama.log"

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(OLLAMA_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print(f"Ollama server started (PID {ollama_server.pid}).")

ollama_ready = False
for attempt in range(15):
    time.sleep(1)
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=5,
        )
        print("Ollama server: RUNNING")
        print("HTTP status:", response.status)
        ollama_ready = True
        break
    except Exception:
        pass

if not ollama_ready:
    raise RuntimeError(
        "Ollama server did not start.\n\n" + open(OLLAMA_LOG).read()
    )

## Step 5 - Load and Verify `qwen2.5:7b`

This step downloads the `qwen2.5:7b` model into Ollama. The name identifies the Qwen 2.5 model family and its approximately seven-billion-parameter size. The initial download can take several minutes, but rerunning the cell in the same Colab runtime can reuse the downloaded model.

The cell then:

1. Lists the models available to Ollama.
2. Sends a short test prompt to verify inference.
3. Displays active model processing information.

The expected test response is `Ollama GPU test successful.` After the inference, inspect the `PROCESSOR` column produced by `ollama ps`; it should indicate GPU use. A CPU assignment will usually work much more slowly.

In [ ]:
!ollama pull qwen2.5:7b
!ollama list
!ollama run qwen2.5:7b "Reply with exactly: Ollama GPU test successful."
!ollama ps

## Step 6 - Configure and Start Project0

This step supplies Project0's runtime configuration and starts the Dashboard as a background process. It creates a unique run identifier, prepares a directory for diagnostic artifacts, stops a Dashboard left by an earlier execution, and clears the previous Dashboard log.

The notebook configures:

- Ollama as the reasoning provider
- `qwen2.5:7b` as the Documentation Agent reasoning model
- OpenAlex, Crossref, arXiv, and OpenReview as Research Agent sources
- DEBUG logging for development and diagnostics

Project0 listens inside Colab at `http://127.0.0.1:8001`. The cell waits for that address to respond before reporting `Project0 Dashboard: RUNNING`. Runtime messages are written to `/content/project0_dashboard.log`.

If startup fails, the cell prints the Dashboard log. Resolve the reported error and rerun this step before continuing to the public-link step.

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path
from datetime import datetime

# ============================================================
# Project0 Colab Debug Configuration
# ============================================================

PROJECT0_ROOT = "/content/project0"
PROJECT0_LOG = "/content/project0_dashboard.log"

# Optional artifact directory for this Colab session
DEBUG_ARTIFACT_ROOT = Path("/content/project0_debug_artifacts")
DEBUG_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

# Unique identifier for this test iteration
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Project0 debug iteration: {RUN_ID}")


# ============================================================
# Stop any previous Project0 dashboard process
# ============================================================

subprocess.run(
    [
        "pkill",
        "-f",
        "project0.dashboard.dashboard_app",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(1)


# ============================================================
# Clear the dashboard log
#
# This preserves the same workflow used locally:
# every test iteration begins with a clean terminal log.
# ============================================================

Path(PROJECT0_LOG).write_text("")


# ============================================================
# Project0 runtime configuration
# ============================================================

os.environ["PROJECT0_REASONING_PROVIDER"] = "ollama"
os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"] = "qwen2.5:7b"

os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"] = (
    "openalex,crossref,arxiv,openreview"
)

os.environ["PROJECT0_LOG_LEVEL"] = "DEBUG"


print("\nProject0 configuration:")
print(
    "  Reasoning provider:",
    os.environ["PROJECT0_REASONING_PROVIDER"],
)
print(
    "  Ollama model:",
    os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"],
)
print(
    "  Research sources:",
    os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"],
)
print(
    "  Log level:",
    os.environ["PROJECT0_LOG_LEVEL"],
)


# ============================================================
# Start Project0 dashboard
# ============================================================

log_file = open(
    PROJECT0_LOG,
    "w",
    buffering=1,
)

dashboard = subprocess.Popen(
    [
        "python",
        "-m",
        "project0.dashboard.dashboard_app",
    ],
    cwd=PROJECT0_ROOT,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(
    f"\nProject0 process started "
    f"(PID {dashboard.pid})."
)

print("Waiting for dashboard...")


# ============================================================
# Wait for dashboard
# ============================================================

dashboard_ready = False

for attempt in range(15):
    time.sleep(1)

    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:8001",
            timeout=2,
        )

        print("Project0 Dashboard: RUNNING")
        print("HTTP status:", response.status)

        dashboard_ready = True
        break

    except Exception:
        pass


if not dashboard_ready:
    print("Project0 dashboard did not start.")

    print("\n===== Project0 log =====")

    log_text = Path(PROJECT0_LOG).read_text(
        errors="replace"
    )

    print(log_text)



## Step 7 - Open the Project0 Dashboard

The Dashboard is running inside Colab, so your browser cannot normally reach its private `127.0.0.1:8001` address. This step installs Cloudflare's `cloudflared` utility and creates a temporary encrypted tunnel from a public HTTPS address to that local Dashboard. No Cloudflare account or credentials are required.

If you rerun this cell, it stops the tunnel previously created by the cell before opening a new one. When startup succeeds, Colab displays an **Open Project0 Dashboard** link. Open it in a new browser tab.

The generated `trycloudflare.com` URL:

- Exists only while the Colab runtime and tunnel process remain active.
- Changes when a new tunnel is created.
- Provides access to the Dashboard to anyone who has the link, so do not share it unnecessarily.

If no URL appears, confirm that Step 6 reported a running Dashboard and then rerun this cell. The tunnel only carries browser traffic; Ollama continues to run privately inside Colab.

In [ ]:
from pathlib import Path
import re
import subprocess
import time
from IPython.display import display, HTML

CLOUDFLARED_PATH = Path("/usr/local/bin/cloudflared")

# ---------------------------------------------------------------------
# Install cloudflared if necessary.
# ---------------------------------------------------------------------

if not CLOUDFLARED_PATH.exists():
    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(CLOUDFLARED_PATH),
        ],
        check=True,
    )

    subprocess.run(
        ["chmod", "+x", str(CLOUDFLARED_PATH)],
        check=True,
    )

print(
    subprocess.run(
        [str(CLOUDFLARED_PATH), "--version"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

# ---------------------------------------------------------------------
# Stop a tunnel previously started by this notebook cell, if present.
# This makes the cell safe to rerun.
# ---------------------------------------------------------------------

if "cloudflared_process" in globals():
    if cloudflared_process.poll() is None:
        print("Stopping existing Project0 tunnel...")
        cloudflared_process.terminate()

        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            cloudflared_process.kill()
            cloudflared_process.wait()

# ---------------------------------------------------------------------
# Start a Cloudflare Quick Tunnel to Project0.
# ---------------------------------------------------------------------

print("Starting Project0 Cloudflare tunnel...")

cloudflared_process = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        "http://127.0.0.1:8001",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
deadline = time.time() + 45

while time.time() < deadline:
    line = cloudflared_process.stdout.readline()

    if line:
        print(line.rstrip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line,
        )

        if match:
            tunnel_url = match.group(0)
            break

    if cloudflared_process.poll() is not None:
        break

if tunnel_url is None:
    if cloudflared_process.poll() is None:
        cloudflared_process.terminate()

    raise RuntimeError(
        "Cloudflare tunnel did not provide a dashboard URL. "
        "Verify that Project0 is running on 127.0.0.1:8001 "
        "and rerun this cell."
    )

# ---------------------------------------------------------------------
# Display the Dashboard link.
# ---------------------------------------------------------------------

print()
print("Project0 Dashboard ready:")
print(tunnel_url)

display(
    HTML(
        f"""
        <p>
          <a href="{tunnel_url}"
             target="_blank"
             rel="noopener noreferrer"
             style="font-size:18px;font-weight:bold;">
             Open Project0 Dashboard
          </a>
        </p>
        """
    )
)



## Step 8 - Stop Project0 (Optional)

Use this optional cleanup step when you are finished with the Project0 Dashboard.

The `STOP_PROJECT0` checkbox in the following cell defaults to **False**. Therefore, **Run all** safely skips cleanup and leaves both the Dashboard and its public link running. The message `Project0 cleanup skipped.` confirms this behavior.

When you are ready to stop the services:

1. Enable the `STOP_PROJECT0` checkbox.
2. Run the Step 8 code cell manually.

The cleanup stops the Cloudflare Quick Tunnel created in Step 7 and the Project0 Dashboard process created in Step 6. It does not delete the cloned repository, downloaded model, logs, or other files in the current runtime.

To end the entire Colab session and remove all temporary files under `/content`, use **Runtime → Disconnect and delete runtime**.

In [ ]:
import subprocess

STOP_PROJECT0 = False  # @param {type:"boolean"}

if not STOP_PROJECT0:
    print("Project0 cleanup skipped.")
else:
    # Stop the Cloudflare Quick Tunnel.
    if (
        "cloudflared_process" in globals()
        and cloudflared_process.poll() is None
    ):
        print("Stopping Project0 Cloudflare tunnel...")
        cloudflared_process.terminate()

        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            print(
                "Cloudflare tunnel did not stop normally; "
                "terminating it..."
            )
            cloudflared_process.kill()
            cloudflared_process.wait()

        print("Cloudflare tunnel stopped.")
    else:
        print("Cloudflare tunnel is not running.")

    # Stop the Project0 Dashboard.
    if "dashboard" in globals() and dashboard.poll() is None:
        print("Stopping Project0 Dashboard...")
        dashboard.terminate()

        try:
            dashboard.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print(
                "Project0 did not stop normally; terminating it..."
            )
            dashboard.kill()
            dashboard.wait()

        print("Project0 Dashboard stopped.")
    else:
        print("Project0 Dashboard is not running.")

    print()
    print("Project0 Colab session cleanup complete.")

